# Pengaruh Ukuran Input terhadap *Instance Segmentation* Objek Dalam Ruangan

**Ringkasan.** Tiga model *instance segmentation* dari tiga keluarga arsitektur berbeda (`YOLO26s-seg`, `Mask R-CNN R50-FPN v2`, dan `RF-DETR-Seg Small`) dilatih pada dataset *Panoptic Indoor Segmentation* yang diunduh lewat Roboflow API pada tiga variasi ukuran input: `320`, `640`, dan `960` piksel. Evaluasi di notebook ini difokuskan pada *split* `valid` untuk memantau respons ukuran input dan memilih *checkpoint* terbaik sebelum pengujian akhir pada *test set* di notebook terpisah.

## Tujuan dan Hipotesis

**Tujuan**
1. Mengukur perubahan *mask* mAP terhadap variasi ukuran input (`320`, `640`, `960` piksel) pada setiap model.
2. Membandingkan perilaku tiga keluarga arsitektur (*one-stage* CNN, *two-stage* CNN, dan *transformer*) ketika resolusi input dinaikkan.
3. Menyiapkan *checkpoint* terbaik, log pelatihan, dan manifes *handoff* untuk proses *inference* pada *split* `test`.

**Hipotesis Awal**
- **H1.** Input beresolusi lebih besar menaikkan *mask* mAP, khususnya pada objek berukuran kecil (*small objects*), dengan kompensasi waktu training dan *inference* yang lebih lama.
- **H2.** Pada *split* `valid`, selisih antara `640` dan `960` piksel cenderung tipis karena mayoritas gambar sumbernya berukuran sekitar `640x480` sehingga pembesaran ke `960` tidak menambah detail piksel asli.
- **H3 (diuji di notebook evaluasi).** Pada *test set* foto kamera HP beresolusi tinggi (`3000x4000` piksel), lompatan dari `640` ke `960` piksel diperkirakan jauh lebih terasa karena detail objek kecil masih tersedia di gambar asli.

## Desain Eksperimen

| Model | Keluarga Arsitektur | Mekanisme Prediksi *Mask* |
|---|---|---|
| `YOLO26s-seg` | CNN *one-stage* | Kombinasi koefisien *instance* dan *prototype mask* |
| `Mask R-CNN R50-FPN v2` | CNN *two-stage* | *Mask* biner `28x28` per *Region of Interest* (RoI) |
| `RF-DETR-Seg Small` | *Transformer* (*backbone* DINOv2) | *Mask* berbasis *object query* |

- **Variabel bebas:** Ukuran input `320`, `640`, dan `960` piksel. Khusus `RF-DETR-Seg Small`, arsitekturnya mensyaratkan dimensi kelipatan `24` (*patch size* 12 x 2 *window*), sehingga memakai `312`, `648`, dan `960` piksel sebagai nilai valid terdekat.
- **Variabel kontrol:** *Optimizer*, skema augmentasi, *effective batch size* `16`, dan *random seed* `42`. Training *multi-scale* dimatikan agar resolusi input tetap konsisten di setiap run.
- **Variabel terikat:** `mask mAP@[.5:.95]`, `AP50`, `AP75`, AP berdasarkan skala COCO (`small`, `medium`, `large`), `AR100`, waktu GPU, serta kecepatan *inference* per gambar.
- **Pemilihan *checkpoint*:** Menggunakan skor `valid` terbaik; *split* `test` tidak pernah diakses selama training.

## 1. Konfigurasi


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time

os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "1"  # periksa GPU lewat NVML agar CUDA belum aktif sebelum fork

# --- Roboflow Dataset ---
ROBOFLOW_WORKSPACE = "fasel"
ROBOFLOW_PROJECT = "panoptic-indoor-segmentation-zsgy2"
ROBOFLOW_VERSION = 1

# --- Penyimpanan Persisten (Google Drive di Colab) ---
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/instance_seg_colab_runs")
LOCAL_WORK = Path("/content/work")
LOCAL_WORK.mkdir(parents=True, exist_ok=True)

IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUT = DRIVE_OUTPUT_DIR
else:
    OUT = LOCAL_WORK / "runs"

EVAL_OUT = OUT / "eval"
RAW_DATA_DIR = LOCAL_WORK / "roboflow_raw"
OUT.mkdir(parents=True, exist_ok=True)
EVAL_OUT.mkdir(parents=True, exist_ok=True)

# --- Matriks Eksperimen ---
SIZES = [320, 640, 960]
RFDETR_RES = {320: 312, 640: 648, 960: 960}  # kelipatan 24 (patch 12 x 2 window)
MODELS = ["yolo26s-seg", "maskrcnn-v2", "rfdetr-seg-small"]

EPOCHS = {"yolo26s-seg": 60, "maskrcnn-v2": 16, "rfdetr-seg-small": 30}
BATCH = {
    "yolo26s-seg": {320: 16, 640: 16, 960: 8},
    "maskrcnn-v2": {320: 8, 640: 4, 960: 2},
    "rfdetr-seg-small": {320: 8, 640: 4, 960: 2},
}
EFFECTIVE_BATCH = 16  # batch_size x gradient_accumulation
SEED, WORKERS = 42, 2

# --- Logging W&B ---
WANDB_PROJECT = "indoor-instance-segmentation"
REQUIRE_WANDB = False  # ubah ke True jika notebook harus berhenti saat WANDB_API_KEY kosong

print({"out_dir": str(OUT), "total_runs": len(MODELS) * len(SIZES), "models": MODELS, "sizes": SIZES})

## 2. Dependensi


In [ ]:
%pip install -q "ultralytics>=8.4.0" "rfdetr[train]>=1.12" "pillow<13" pycocotools roboflow wandb

## 3. Runtime dan GPU


In [ ]:
import torch, torchvision, ultralytics, rfdetr

gpu_lines = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=True,
).stdout.strip().splitlines()
assert len(gpu_lines) >= 1, "GPU tidak terdeteksi. Aktifkan GPU melalui Runtime > Change runtime type."
NUM_GPUS = len(gpu_lines)
print({
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "ultralytics": ultralytics.__version__,
    "rfdetr": getattr(rfdetr, "__version__", "?"),
    "num_gpus": NUM_GPUS,
    "gpus": gpu_lines,
})

## 4. Dataset, Sanitasi EXIF, dan *Exploratory Data Analysis* (EDA)

**Sumber data.** Dataset *Panoptic Indoor Segmentation* diunduh dalam format *COCO Segmentation* melalui Roboflow API dengan *split* `train` dan `valid` bawaan. Pada pemeriksaan awal ditemukan distribusi kelas yang timpang (*long-tail*) serta beberapa gambar yang dimensi pikselnya pasca-koreksi orientasi EXIF tidak cocok dengan dimensi pada JSON COCO.

**Sanitasi data.** Sebelum EDA dan training berjalan, setiap gambar `train` dan `valid` diperiksa memakai `ImageOps.exif_transpose`. Gambar yang ukurannya menyimpang dari catatan `width` dan `height` di metadata COCO dibuang otomatis agar seluruh poligon tetap sejajar dengan objek.

In [ ]:
from getpass import getpass
from PIL import Image, ImageOps
from roboflow import Roboflow


def get_secret(name, required=True):
    val = os.environ.get(name, "").strip()
    if not val and IN_COLAB:
        try:
            from google.colab import userdata
            val = (userdata.get(name) or "").strip()
        except Exception:
            val = ""
    if not val and required:
        val = getpass(f"Masukkan {name}: ").strip()
    if not val and required:
        raise RuntimeError(f"{name} wajib diisi (Colab Secrets atau input interaktif).")
    return val


def find_root(root, split):
    if (root / split / "_annotations.coco.json").is_file():
        return root
    hits = [p.parent.parent for p in root.rglob("_annotations.coco.json") if p.parent.name == split]
    return hits[0] if hits else None


rf_key = get_secret("ROBOFLOW_API_KEY", required=True)
if not find_root(RAW_DATA_DIR, "train"):
    # Roboflow SDK melewati unduhan jika folder `location` sudah ada dan `overwrite=False`.
    if RAW_DATA_DIR.exists():
        shutil.rmtree(RAW_DATA_DIR)
    rf = Roboflow(api_key=rf_key)
    proj = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    rf_ds = proj.version(ROBOFLOW_VERSION).download("coco-segmentation", location=str(RAW_DATA_DIR), overwrite=True)
    SRC_ROOT = find_root(Path(rf_ds.location), "train") or find_root(RAW_DATA_DIR, "train")
else:
    SRC_ROOT = find_root(RAW_DATA_DIR, "train")
assert SRC_ROOT and (SRC_ROOT / "valid/_annotations.coco.json").is_file(), f"Dataset train/valid tidak ditemukan di {RAW_DATA_DIR}"

# Sanitasi: dimensi gambar pasca-koreksi EXIF harus identik dengan metadata COCO.
CLEAN_ROOT, DROPPED = LOCAL_WORK / "data_clean", {}
for split in ("train", "valid"):
    src = SRC_ROOT / split
    d = json.loads((src / "_annotations.coco.json").read_text(encoding="utf-8"))
    bad = set()
    for im in d["images"]:
        with Image.open(src / im["file_name"]) as f:
            if ImageOps.exif_transpose(f).size != (int(im["width"]), int(im["height"])):
                bad.add(im["id"])
    DROPPED[split] = sorted(im["file_name"] for im in d["images"] if im["id"] in bad)
    d["images"] = [im for im in d["images"] if im["id"] not in bad]
    d["annotations"] = [a for a in d["annotations"] if a["image_id"] not in bad]
    dst = CLEAN_ROOT / split
    dst.mkdir(parents=True, exist_ok=True)
    for im in d["images"]:
        target = dst / im["file_name"]
        if not target.exists():
            try:
                os.symlink((src / im["file_name"]).resolve(), target)
            except OSError:
                shutil.copy2((src / im["file_name"]).resolve(), target)
    (dst / "_annotations.coco.json").write_text(json.dumps(d), encoding="utf-8")

DATASET_ROOT = CLEAN_ROOT
print("Gambar dibuang (dimensi pasca-EXIF != anotasi):", {k: len(v) for k, v in DROPPED.items()}, DROPPED)

TRAIN_COCO = json.loads((DATASET_ROOT / "train/_annotations.coco.json").read_text(encoding="utf-8"))
VALID_COCO = json.loads((DATASET_ROOT / "valid/_annotations.coco.json").read_text(encoding="utf-8"))
CATEGORY_LIST = sorted((c for c in TRAIN_COCO["categories"] if int(c["id"]) != 0), key=lambda c: int(c["id"]))
CATEGORY_IDS = [int(c["id"]) for c in CATEGORY_LIST]
CATEGORY_NAMES = [c["name"] for c in CATEGORY_LIST]
NAME_TO_ID = dict(zip(CATEGORY_NAMES, CATEGORY_IDS))
for name, d in (("train", TRAIN_COCO), ("valid", VALID_COCO)):
    print(f"{name:5s}: {len(d['images'])} gambar, {len(d['annotations'])} instance")
print("Total kelas foreground:", len(CATEGORY_NAMES))

### 4.1 Ukuran Gambar, Kepadatan *Instance*, dan Skala *Bounding Box*

Bagian ini memeriksa tiga faktor yang berhubungan langsung dengan pengujian ukuran input:
1. **Resolusi gambar sumber:** Menentukan apakah perbesaran input dari `640` ke `960` piksel memanfaatkan piksel asli atau sekadar *upscaling*.
2. **Kepadatan *instance* per gambar:** Menggambarkan tingkat keramaian objek dalam satu adegan (*scene clutter*).
3. **Skala objek COCO (*small* `< 32^2` px, *medium* `32^2-96^2` px, *large* `> 96^2` px):** Memetakan porsi objek kecil yang paling berisiko hilang pada resolusi `320` piksel.

In [ ]:
from collections import Counter
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Polygon

train_categories = sorted((int(c["id"]), c["name"]) for c in TRAIN_COCO["categories"])
valid_categories = sorted((int(c["id"]), c["name"]) for c in VALID_COCO["categories"])
assert train_categories == valid_categories, "Peta kategori train dan valid berbeda."
assert len({int(i["id"]) for i in TRAIN_COCO["images"]}) == len(TRAIN_COCO["images"])
assert len({int(i["id"]) for i in VALID_COCO["images"]}) == len(VALID_COCO["images"])

split_data = {"train": TRAIN_COCO, "valid": VALID_COCO}
dimension_rows, object_rows, box_rows = [], [], []
counts_by_split = {}
for split, data in split_data.items():
    per_image = Counter(int(a["image_id"]) for a in data["annotations"] if int(a["category_id"]) != 0)
    counts_by_split[split] = Counter(int(a["category_id"]) for a in data["annotations"])
    for image in data["images"]:
        w, h = int(image["width"]), int(image["height"])
        dimension_rows.append({
            "split": split, "width": w, "height": h,
            "aspect_ratio": w / h, "res_group": f"{w}x{h}",
        })
        object_rows.append({"split": split, "instances_per_image": per_image[int(image["id"])]})
    image_by_id = {int(i["id"]): i for i in data["images"]}
    for ann in data["annotations"]:
        if int(ann["category_id"]) == 0:
            continue
        x, y, bw, bh = map(float, ann["bbox"])
        assert bw > 0 and bh > 0, f"Bounding box tidak valid pada {split}, anotasi {ann['id']}"
        im = image_by_id[int(ann["image_id"])]
        area_px = float(ann.get("area", bw * bh))
        scale = "small (<32^2)" if area_px < 32 ** 2 else ("medium (32^2-96^2)" if area_px <= 96 ** 2 else "large (>96^2)")
        box_rows.append({
            "split": split,
            "bbox_area_fraction": (bw * bh) / (int(im["width"]) * int(im["height"])),
            "coco_scale": scale,
        })

dimensions = pd.DataFrame(dimension_rows)
objects_per_image = pd.DataFrame(object_rows)
bbox_fractions = pd.DataFrame(box_rows)

print("Ringkasan dimensi gambar dan aspect ratio:")
display(dimensions.groupby("split")[["width", "height", "aspect_ratio"]].agg(["min", "median", "max"]).round(3))
print("Ringkasan jumlah instance per gambar:")
display(objects_per_image.groupby("split")["instances_per_image"].describe().round(2))
print("Ringkasan fraksi luas bounding box:")
display(bbox_fractions.groupby("split")["bbox_area_fraction"].describe(percentiles=[.25, .5, .75]).round(4))

# Visualisasi EDA 3 panel: resolusi gambar, kepadatan instance, dan skala COCO.
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

top_res = dimensions["res_group"].value_counts().head(5).index
res_plot = dimensions.assign(res_bucket=np.where(dimensions["res_group"].isin(top_res), dimensions["res_group"], "lainnya"))
res_pct = pd.crosstab(res_plot["res_bucket"], res_plot["split"], normalize="columns") * 100
res_pct.plot.barh(ax=axes[0], color=["#4C78A8", "#F58518"])
axes[0].set_title("Proporsi Resolusi Gambar (%)")
axes[0].set_xlabel("Persentase gambar (%)"); axes[0].set_ylabel(""); axes[0].grid(axis="x", alpha=.25)

for sp, col in (("train", "#4C78A8"), ("valid", "#F58518")):
    vals = objects_per_image[objects_per_image["split"] == sp]["instances_per_image"]
    axes[1].hist(vals, bins=20, alpha=.55, density=True, label=f"{sp} (median={vals.median():.0f})", color=col)
axes[1].set_title("Distribusi Kepadatan Instance per Gambar")
axes[1].set_xlabel("Jumlah instance"); axes[1].set_ylabel("Densitas"); axes[1].legend(); axes[1].grid(alpha=.25)

scale_order = ["small (<32^2)", "medium (32^2-96^2)", "large (>96^2)"]
scale_pct = pd.crosstab(bbox_fractions["coco_scale"], bbox_fractions["split"], normalize="columns").reindex(scale_order) * 100
scale_pct.plot.bar(ax=axes[2], rot=15, color=["#4C78A8", "#F58518"])
axes[2].set_title("Proporsi Skala Objek COCO (%)")
axes[2].set_ylabel("Persentase instance (%)"); axes[2].set_xlabel(""); axes[2].grid(axis="y", alpha=.25)

plt.tight_layout(); plt.show()

### 4.2 Distribusi Kelas dan Ketimpangan (*Class Imbalance*)

Dataset lingkungan dalam ruangan memiliki sebaran *long-tail*: kelas latar dan furnitur umum seperti dinding atau kursi muncul ribuan kali, sedangkan benda kecil tertentu hanya muncul beberapa kali. Kelas yang sangat jarang muncul pada `valid` membuat nilai AP per kelas pada kelompok tersebut memiliki varians tinggi.

In [ ]:
class_rows = [
    {"category_id": cid, "class": name,
     "train_instances": counts_by_split["train"].get(cid, 0),
     "valid_instances": counts_by_split["valid"].get(cid, 0)}
    for cid, name in train_categories if cid != 0
]
class_table = pd.DataFrame(class_rows).sort_values("train_instances", ascending=False)
zero_valid = int((class_table["valid_instances"] == 0).sum())
few_valid = int((class_table["valid_instances"] <= 5).sum())
print(f"Total kelas: {len(class_table)} | Tanpa instance valid: {zero_valid} | <= 5 instance valid: {few_valid}")

print("15 kelas dengan jumlah instance terbanyak (train):")
display(class_table.head(15))
print("10 kelas dengan jumlah instance tersedikit (train):")
display(class_table.tail(10).sort_values("train_instances"))

fig, ax = plt.subplots(figsize=(10, 15))
ordered = class_table.sort_values("train_instances")
y_pos = np.arange(len(ordered))
ax.barh(y_pos, ordered["train_instances"], color="#4C78A8", label="train")
ax.barh(y_pos, ordered["valid_instances"], color="#F58518", alpha=.85, label="valid")
ax.set_yticks(y_pos); ax.set_yticklabels(ordered["class"], fontsize=8)
ax.set_xlabel("Jumlah instance"); ax.set_title("Distribusi Jumlah Instance per Kelas Foreground (Train vs Valid)")
ax.legend(); ax.grid(axis="x", alpha=.25)
plt.tight_layout(); plt.show()

### 4.3 Audit Visual Anotasi

Empat sampel deterministik dari `train` dan `valid` ditampilkan bersama poligon *ground truth* setelah koreksi EXIF untuk memastikan keselarasan batas objek dan gambar.

In [ ]:
id_to_name = dict(zip(CATEGORY_IDS, CATEGORY_NAMES))
rng = np.random.default_rng(SEED)
samples = []
for split, data in split_data.items():
    per_image = Counter(int(a["image_id"]) for a in data["annotations"])
    candidates = [im for im in data["images"] if 2 <= per_image[int(im["id"])] <= 12] or data["images"]
    picks = rng.choice(len(candidates), size=min(2, len(candidates)), replace=False)
    samples.extend((split, candidates[int(i)]) for i in picks)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
for ax, (split, image_info) in zip(axes.flat, samples):
    with Image.open(DATASET_ROOT / split / image_info["file_name"]) as im:
        rgb = np.asarray(ImageOps.exif_transpose(im).convert("RGB"))
    anns = [a for a in split_data[split]["annotations"]
            if int(a["image_id"]) == int(image_info["id"]) and int(a["category_id"]) != 0]
    ax.imshow(rgb)
    for j, ann in enumerate(anns):
        cname = id_to_name.get(int(ann["category_id"]), "?")
        for k, polygon in enumerate(ann["segmentation"]):
            points = np.asarray(polygon, dtype=float).reshape(-1, 2)
            if len(points) >= 3:
                color = plt.cm.tab20(j % 20)
                ax.add_patch(Polygon(points, closed=True, facecolor=color, edgecolor=color, alpha=.25, linewidth=1.3))
                if k == 0:
                    cx, cy = points.mean(axis=0)
                    ax.text(cx, cy, cname, fontsize=7, color="white",
                            bbox=dict(boxstyle="round,pad=0.15", facecolor="black", alpha=.55, lw=0))
    ax.set_title(f"{split}: {image_info['file_name']} ({rgb.shape[1]}x{rgb.shape[0]}, {len(anns)} instance)", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Pencatatan Eksperimen (*Weights & Biases*)

Jika `WANDB_API_KEY` tersedia pada Colab Secrets atau variabel lingkungan, metrik per *epoch* dicatat otomatis ke proyek W&B. Bila kunci tidak disetel dan `REQUIRE_WANDB = False`, proses training tetap berjalan menggunakan log lokal `results.csv` dan TensorBoard.

In [ ]:
import wandb

wb_key = get_secret("WANDB_API_KEY", required=REQUIRE_WANDB)
if wb_key:
    os.environ["WANDB_API_KEY"] = wb_key
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
    assert wandb.login(key=wb_key, relogin=True), "Login W&B gagal."
    print("W&B aktif:", WANDB_PROJECT)
else:
    os.environ.pop("WANDB_API_KEY", None)
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B tidak aktif (WANDB_API_KEY kosong); log lokal tetap tersimpan di", OUT)

## 6. *Preprocessing* Data per Model

Tiap arsitektur membutuhkan format anotasi dan cara *resizing* gambar yang berbeda:

| Model | Format Label | Skema *Resize* Input | Augmentasi Saat Training |
|---|---|---|---|
| **YOLO26s-seg** | Poligon ternormalisasi `[0, 1]` di file `.txt` (indeks `0..73`) | *Letterbox* ke kanvas persegi `size x size` dengan *padding* abu-abu `RGB 114` | Bawaan Ultralytics: *mosaic*, *HSV jitter*, *scale*, dan *horizontal flip* |
| **Mask R-CNN v2** | Poligon COCO di-*decode* ke *mask* biner + `masks_to_boxes` | `GeneralizedRCNNTransform` menskalakan sisi terpanjang ke `size` tanpa *padding* persegi | *Horizontal flip* (`p = 0.5`) dan *brightness/contrast jitter* |
| **RF-DETR-Seg Small** | Struktur direktori COCO Roboflow (`train/`, `valid/`) | *Resize* ke persegi tetap kelipatan 24 (`312`, `648`, `960`) pada mode *single-scale* | Augmentasi bawaan RF-DETR pada resolusi tetap |

Sel di bawah mengekspor label YOLO ke `/content/work/yolo_format` dan menautkan struktur COCO untuk RF-DETR ke `/content/work/rfdetr_data`.

In [ ]:
import yaml

YOLO_ROOT = LOCAL_WORK / "yolo_format"
cat_to_yolo = {cid: i for i, cid in enumerate(CATEGORY_IDS)}
for split in ("train", "valid"):
    src = DATASET_ROOT / split
    raw = json.loads((src / "_annotations.coco.json").read_text(encoding="utf-8"))
    (YOLO_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)
    anns = {}
    for a in raw["annotations"]:
        if int(a["category_id"]) != 0:
            anns.setdefault(int(a["image_id"]), []).append(a)
    for im in raw["images"]:
        dst = YOLO_ROOT / split / "images" / im["file_name"]
        if not dst.exists():
            try:
                os.symlink((src / im["file_name"]).resolve(), dst)
            except OSError:
                shutil.copy2((src / im["file_name"]).resolve(), dst)
        rows = []
        for a in anns.get(int(im["id"]), []):
            for poly in a["segmentation"]:
                xy = [min(1.0, max(0.0, float(v) / (im["width"] if j % 2 == 0 else im["height"]))) for j, v in enumerate(poly)]
                if len(xy) >= 6:
                    rows.append(f"{cat_to_yolo[int(a['category_id'])]} " + " ".join(f"{v:.6f}" for v in xy))
        (YOLO_ROOT / split / "labels" / (Path(im["file_name"]).stem + ".txt")).write_text("\n".join(rows), encoding="utf-8")

YOLO_YAML = YOLO_ROOT / "dataset.yaml"
yaml.safe_dump(
    {"path": str(YOLO_ROOT), "train": "train/images", "val": "valid/images", "names": dict(enumerate(CATEGORY_NAMES))},
    YOLO_YAML.open("w"), sort_keys=False,
)

# RF-DETR mensyaratkan direktori train, valid, dan test; test ditautkan ke valid karena pengujian asli dijalankan di notebook terpisah.
RFDETR_ROOT = LOCAL_WORK / "rfdetr_data"
RFDETR_ROOT.mkdir(exist_ok=True)
for split, src in (("train", DATASET_ROOT / "train"), ("valid", DATASET_ROOT / "valid"), ("test", DATASET_ROOT / "valid")):
    if not (RFDETR_ROOT / split).exists():
        try:
            os.symlink(src.resolve(), RFDETR_ROOT / split)
        except OSError:
            shutil.copytree(src.resolve(), RFDETR_ROOT / split)

print("Siap:", YOLO_YAML, RFDETR_ROOT)

### 6.1 Verifikasi Visual *Preprocessing* (`320`, `640`, dan `960`)

Sel berikut menampilkan pratinjau hasil konversi label YOLO `.txt`, *letterbox* `320x320`, transformasi Mask R-CNN (`640`), serta kanvas persegi `RF-DETR-Seg Small` (`648x648`) pada satu sampel `train` untuk memastikan koordinat *mask* tidak bergeser sebelum pelatihan dimulai.

In [ ]:
from pycocotools import mask as mu

sample_im = next(
    im for im in TRAIN_COCO["images"]
    if 3 <= sum(1 for a in TRAIN_COCO["annotations"] if int(a["image_id"]) == int(im["id"]) and int(a["category_id"]) != 0) <= 10
)
sample_anns = [a for a in TRAIN_COCO["annotations"] if int(a["image_id"]) == int(sample_im["id"]) and int(a["category_id"]) != 0]
with Image.open(DATASET_ROOT / "train" / sample_im["file_name"]) as f:
    src_rgb = np.asarray(ImageOps.exif_transpose(f).convert("RGB"))
sh, sw = src_rgb.shape[:2]

# 1) Simulasi letterbox YOLO + baca ulang poligon dari berkas .txt hasil konversi
y_size = 320
y_scale = min(y_size / sw, y_size / sh)
ynw, ynh = max(1, round(sw * y_scale)), max(1, round(sh * y_scale))
yx0, yy0 = (y_size - ynw) // 2, (y_size - ynh) // 2
yolo_canvas = np.full((y_size, y_size, 3), 114, dtype=np.uint8)
yolo_canvas[yy0:yy0 + ynh, yx0:yx0 + ynw] = cv2.resize(src_rgb, (ynw, ynh), interpolation=cv2.INTER_LINEAR)
yolo_txt = (YOLO_ROOT / "train/labels" / (Path(sample_im["file_name"]).stem + ".txt")).read_text().splitlines()

# 2) Simulasi preprocessing Mask R-CNN v2 (sisi terpanjang = 640, horizontal flip + jitter + nearest mask)
m_size = 640
m_scale = m_size / max(sh, sw)
mnw, mnh = max(1, round(sw * m_scale)), max(1, round(sh * m_scale))
mrcnn_img = np.clip(cv2.resize(src_rgb[:, ::-1], (mnw, mnh), interpolation=cv2.INTER_LINEAR).astype(np.float32) * 1.15 - 10, 0, 255).astype(np.uint8)

# 3) Simulasi resize persegi RF-DETR-Seg Small (648x648)
r_size = RFDETR_RES[640]
rfdetr_img = cv2.resize(src_rgb, (r_size, r_size), interpolation=cv2.INTER_LINEAR)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))

axes[0].imshow(src_rgb)
for j, a in enumerate(sample_anns):
    for poly in a["segmentation"]:
        pts = np.asarray(poly, dtype=float).reshape(-1, 2)
        if len(pts) >= 3:
            axes[0].add_patch(Polygon(pts, closed=True, facecolor=plt.cm.tab20(j % 20), edgecolor=plt.cm.tab20(j % 20), alpha=.28, lw=1.2))
axes[0].set_title(f"1. Asli COCO ({sw}x{sh})")

axes[1].imshow(yolo_canvas)
for j, line in enumerate(yolo_txt):
    nums = list(map(float, line.split()[1:]))
    pts = np.asarray(nums, dtype=float).reshape(-1, 2)
    pts[:, 0] = pts[:, 0] * ynw + yx0
    pts[:, 1] = pts[:, 1] * ynh + yy0
    axes[1].add_patch(Polygon(pts, closed=True, facecolor=plt.cm.tab20(j % 20), edgecolor=plt.cm.tab20(j % 20), alpha=.32, lw=1.2))
axes[1].set_title(f"2. YOLO Letterbox ({y_size}x{y_size}, pad=114)")

axes[2].imshow(mrcnn_img)
for j, a in enumerate(sample_anns):
    rle = mu.merge(mu.frPyObjects(a["segmentation"], sh, sw))
    m_flip = mu.decode(rle)[:, ::-1]
    m_res = cv2.resize(m_flip, (mnw, mnh), interpolation=cv2.INTER_NEAREST)
    ov = np.zeros((mnh, mnw, 4), dtype=float); ov[..., :3] = plt.cm.tab20(j % 20)[:3]; ov[..., 3] = m_res * .35
    axes[2].imshow(ov)
axes[2].set_title(f"3. Mask R-CNN ({mnw}x{mnh}, flip+jitter)")

axes[3].imshow(rfdetr_img)
for j, a in enumerate(sample_anns):
    for poly in a["segmentation"]:
        pts = np.asarray(poly, dtype=float).reshape(-1, 2)
        pts[:, 0] *= r_size / sw
        pts[:, 1] *= r_size / sh
        if len(pts) >= 3:
            axes[3].add_patch(Polygon(pts, closed=True, facecolor=plt.cm.tab20(j % 20), edgecolor=plt.cm.tab20(j % 20), alpha=.28, lw=1.2))
axes[3].set_title(f"4. RF-DETR Square ({r_size}x{r_size})")

for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

summary_prep = []
for sz in SIZES:
    ys = min(sz / sw, sz / sh)
    ynw_s, ynh_s = round(sw * ys), round(sh * ys)
    ms = sz / max(sh, sw)
    summary_prep.append({
        "target_size": sz,
        "yolo_canvas": f"{sz}x{sz}",
        "yolo_content_hw": f"{ynh_s}x{ynw_s}",
        "yolo_pad_lr_tb": f"({(sz-ynw_s)//2}, {(sz-ynh_s)//2})",
        "mrcnn_hw": f"{round(sh*ms)}x{round(sw*ms)}",
        "rfdetr_hw": f"{RFDETR_RES[sz]}x{RFDETR_RES[sz]}",
    })
display(pd.DataFrame(summary_prep))

## 7. Fungsi Bersama dan Konfigurasi Pelatihan

| Parameter | YOLO26s-seg | Mask R-CNN v2 | RF-DETR-Seg Small |
|---|---|---|---|
| *Pretrained weights* | COCO | COCO | COCO |
| *Optimizer* & *learning rate* | AdamW, `1e-3`, *cosine* | AdamW, `1e-4`, *warmup* + *cosine* | AdamW, `1e-4` |
| *Epoch* | 60 (*early stopping* 15) | 16 | 30 (*early stopping* 10) |
| *Batch* (`320/640/960`) | `16 / 16 / 8` | `8 / 4 / 2` | `8 / 4 / 2` |
| *Effective batch size* | 16 | 16 | 16 |
| Presisi | AMP FP16 | AMP FP16 | AMP |

Jumlah *epoch* menyesuaikan laju konvergensi tiap arsitektur: `Mask R-CNN` yang menggunakan *backbone* R50-FPN terlatih konvergen lebih cepat daripada detektor *one-stage* maupun *transformer* yang masih mengadaptasi *object query*.

In [ ]:
import contextlib, csv, io, math, random, traceback
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

METRICS = ["mask_mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR100"]


def read_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def load_rgb(path):
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert("RGB"))


def decode(seg, h, w):
    rle = mu.merge(mu.frPyObjects(seg, h, w)) if isinstance(seg, list) else (
        mu.frPyObjects(seg, h, w) if isinstance(seg["counts"], list) else seg
    )
    return mu.decode(rle)


def remap_gt(src_json, dst_json, n2i):
    d = read_json(src_json)
    old = {int(c["id"]): c["name"] for c in d["categories"]}
    names = {old.get(int(a["category_id"])) for a in d["annotations"]}
    d["annotations"] = [
        dict(a, category_id=n2i[old[int(a["category_id"])]], iscrowd=int(a.get("iscrowd", 0)))
        for a in d["annotations"] if old.get(int(a["category_id"])) in n2i
    ]
    d["categories"] = [{"id": i, "name": n, "supercategory": "indoor"} for n, i in n2i.items()]
    Path(dst_json).write_text(json.dumps(d), encoding="utf-8")
    return sorted(n for n in names if n and n not in n2i)


def to_record(image_id, cat_id, score, mask):
    rle = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return {"image_id": int(image_id), "category_id": int(cat_id), "segmentation": rle, "score": float(score)}


def predict_set(run, gt_json, image_root, max_det=100):
    recs = []
    for im in read_json(gt_json)["images"]:
        rgb = load_rgb(Path(image_root) / im["file_name"])
        if rgb.shape[:2] != (im["height"], im["width"]):
            raise ValueError(f"Ukuran {im['file_name']} {rgb.shape[:2]} != anotasi {(im['height'], im['width'])}")
        for c, s, m in sorted(run(rgb), key=lambda t: -t[1])[:max_det]:
            if m.any():
                recs.append(to_record(im["id"], c, s, m))
    return recs


def coco_mask_eval(gt_json, records, img_ids=None):
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(gt_json))
        if not records:
            return dict.fromkeys(METRICS, 0.0)
        ev = COCOeval(gt, gt.loadRes(records), "segm")
        if img_ids is not None:
            ev.params.imgIds = list(img_ids)
        ev.evaluate(); ev.accumulate(); ev.summarize()
    return dict(zip(METRICS, [float(ev.stats[i]) for i in (0, 1, 2, 3, 4, 5, 8)]))


def build_maskrcnn(num_classes, size, pretrained):
    from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights, maskrcnn_resnet50_fpn_v2
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

    m = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1 if pretrained else None,
        weights_backbone=None, box_score_thresh=0.001, box_detections_per_img=100,
    )
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, num_classes + 1)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(m.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes + 1)
    m.transform.min_size, m.transform.max_size = (size,), size
    return m


def maskrcnn_runner(model, ids):
    device = next(model.parameters()).device

    def run(rgb):
        x = torch.from_numpy(rgb.copy()).permute(2, 0, 1).float().div(255).to(device)
        with torch.inference_mode(), torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            o = model([x])[0]
        masks = (o["masks"][:, 0].float() > 0.5).cpu().numpy()
        return [(ids[int(l) - 1], float(s), m) for l, s, m in zip(o["labels"].tolist(), o["scores"].tolist(), masks)]

    return run


def make_predictor(kind, weights, size, n2i):
    if kind == "yolo26s-seg":
        from ultralytics import YOLO
        model = YOLO(str(weights))
        names = model.names

        def run(rgb):
            r = model.predict(
                rgb[..., ::-1].copy(), imgsz=size, conf=0.001, iou=0.7, max_det=100,
                retina_masks=True, half=True, verbose=False, device=0,
            )[0]
            if r.masks is None:
                return []
            masks = r.masks.data.cpu().numpy() > 0.5
            return [(n2i[names[int(c)]], s, m) for c, s, m in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), masks)
                    if names[int(c)] in n2i]

        return run
    if kind == "maskrcnn-v2":
        model = build_maskrcnn(len(n2i), size, pretrained=False)
        model.load_state_dict(torch.load(weights, map_location="cpu", weights_only=False)["model"])
        return maskrcnn_runner(model.cuda().eval(), sorted(n2i.values()))
    if kind == "rfdetr-seg-small":
        from rfdetr import RFDETRSegSmall
        model = RFDETRSegSmall(pretrain_weights=str(weights), resolution=size, trust_checkpoint=True)
        names = model.class_names

        def name_of(cid):
            cid = int(cid)
            if isinstance(names, dict):
                return names.get(cid)
            return names[cid] if 0 <= cid < len(names) else None

        def run(rgb):
            d = model.predict(Image.fromarray(rgb), threshold=0.001)
            if d.mask is None or len(d) == 0:
                return []
            out = []
            for c, s, m in zip(d.class_id, d.confidence, d.mask):
                nm = name_of(c)
                if nm in n2i:
                    out.append((n2i[nm], float(s), np.asarray(m, dtype=bool)))
            return out

        return run
    raise ValueError(kind)

### 7.1 Fungsi Pelatihan per Model

Setiap fungsi mendukung *resume* dari *checkpoint* terakhir serta menulis file `done.json` berisi *hash* SHA-256 saat training selesai. Pada `YOLO26s-seg`, *RAM caching* dimatikan (`cache=False`) untuk menghindari lonjakan memori sistem di ukuran `960` piksel.

In [ ]:
def train_yolo(c):
    from ultralytics import YOLO, settings

    settings.update({"wandb": bool(os.environ.get("WANDB_API_KEY"))})
    out = Path(c["out"])
    last = out / "weights" / "last.pt"
    best = out / "weights" / "best.pt"

    already_finished = False
    can_resume = False
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        if ck.get("epoch", 0) == -1 and best.exists():
            already_finished = True
        else:
            can_resume = True

    if not already_finished:
        model = YOLO(str(last) if can_resume else "yolo26s-seg.pt")
        model.train(
            data=str(YOLO_YAML), epochs=c["epochs"], imgsz=c["size"], batch=c["batch"], nbs=EFFECTIVE_BATCH,
            device=0, workers=WORKERS, optimizer="AdamW", lr0=0.001, lrf=0.01, cos_lr=True, warmup_epochs=3,
            weight_decay=5e-4, patience=15, close_mosaic=10, multi_scale=0, rect=False, cache=False, amp=True,
            seed=SEED, deterministic=False, project=str(out.parent), name=out.name, exist_ok=True,
            resume=can_resume, plots=True, val=True,
        )

    assert best.exists(), f"best.pt tidak ditemukan di {out}"
    (out / "done.json").write_text(json.dumps({
        "model": c["model"], "size": c["run_size"], "weights": str(best), "sha256": file_sha256(best),
    }))


class MaskRCNNTrainSet(torch.utils.data.Dataset):
    '''Augmentasi seragam untuk semua ukuran: flip horizontal + jitter kecerahan/kontras.'''

    def __init__(self, root, label_of):
        d = read_json(Path(root) / "_annotations.coco.json")
        self.root, self.label_of, self.anns = Path(root), label_of, {}
        for ann in d["annotations"]:
            if int(ann["category_id"]) in label_of:
                self.anns.setdefault(ann["image_id"], []).append(ann)
        self.images = [im for im in d["images"] if self.anns.get(im["id"])]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        from torchvision.ops import masks_to_boxes
        im = self.images[i]
        rgb = load_rgb(self.root / im["file_name"]).astype(np.float32)
        h, w = rgb.shape[:2]
        masks, labels = [], []
        for ann in self.anns.get(im["id"], []):
            m = decode(ann["segmentation"], h, w)
            if m.any():
                masks.append(m); labels.append(self.label_of[int(ann["category_id"])])
        if random.random() < 0.5:
            rgb, masks = rgb[:, ::-1], [m[:, ::-1] for m in masks]
        rgb = np.clip(rgb * random.uniform(0.7, 1.3) + random.uniform(-25, 25), 0, 255)
        masks = torch.from_numpy(np.ascontiguousarray(np.stack(masks))) if masks else torch.zeros((0, h, w), dtype=torch.uint8)
        boxes = masks_to_boxes(masks) + torch.tensor([0, 0, 1, 1]) if len(masks) else torch.zeros((0, 4))
        image = torch.from_numpy(np.ascontiguousarray(rgb)).permute(2, 0, 1).div(255)
        return image, {"boxes": boxes.float(), "labels": torch.tensor(labels, dtype=torch.int64), "masks": masks}


def collate_list(batch):
    return tuple(zip(*batch))


def train_maskrcnn(c):
    out = Path(c["out"]); out.mkdir(parents=True, exist_ok=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ids = sorted(NAME_TO_ID.values())
    label_of = {cid: i + 1 for i, cid in enumerate(ids)}
    loader = torch.utils.data.DataLoader(
        MaskRCNNTrainSet(DATASET_ROOT / "train", label_of),
        batch_size=c["batch"], shuffle=True, num_workers=WORKERS, collate_fn=collate_list, drop_last=True,
    )
    model = build_maskrcnn(len(ids), c["size"], pretrained=True).to(device)
    opt = torch.optim.AdamW([q for q in model.parameters() if q.requires_grad], lr=1e-4, weight_decay=1e-4)
    accum = max(1, EFFECTIVE_BATCH // c["batch"])
    batches_per_epoch = len(loader)
    steps_per_epoch = max(1, (batches_per_epoch + accum - 1) // accum)
    total = c["epochs"] * steps_per_epoch
    warm = max(1, total // 20)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))),
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
    start, best = 0, -1.0
    if (out / "last.pt").exists():
        ck = torch.load(out / "last.pt", map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); start, best = ck["epoch"], ck["best"]
    run = None
    if os.environ.get("WANDB_API_KEY"):
        import wandb
        run = wandb.init(project=WANDB_PROJECT, name=f"{out.parent.name}-{out.name}", id=f"{out.parent.name}-{out.name}", resume="allow", config=c)
    valid_json, valid_root = DATASET_ROOT / "valid/_annotations.coco.json", DATASET_ROOT / "valid"
    for epoch in range(start, c["epochs"]):
        t0 = time.time()
        model.train()
        sums, n = {}, 0
        for i, (images, targets) in enumerate(loader):
            images = [x.to(device, non_blocking=True) for x in images]
            targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]
            group_size = min(accum, batches_per_epoch - (i // accum) * accum)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                losses = model(images, targets)
                loss = sum(losses.values())
            scaler.scale(loss / group_size).backward()
            if (i + 1) % accum == 0 or i + 1 == batches_per_epoch:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                scale_before = scaler.get_scale()
                scaler.step(opt)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    sched.step()
                opt.zero_grad(set_to_none=True)
            for k, v in losses.items():
                sums[k] = sums.get(k, 0.0) + float(v)
            sums["loss"] = sums.get("loss", 0.0) + float(loss); n += 1
        model.eval()
        metrics = coco_mask_eval(valid_json, predict_set(maskrcnn_runner(model, ids), valid_json, valid_root))
        row = {
            "epoch": epoch + 1,
            **{f"train/{k}": v / max(n, 1) for k, v in sums.items()},
            **{f"val/{k}": v for k, v in metrics.items()},
            "lr": opt.param_groups[0]["lr"],
            "epoch_seconds": time.time() - t0,
        }
        new = not (out / "results.csv").exists()
        with (out / "results.csv").open("a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row))
            if new: w.writeheader()
            w.writerow(row)
        if run: run.log(row, step=epoch + 1)
        print(f"epoch {epoch + 1}/{c['epochs']} loss={row['train/loss']:.3f} val mask_mAP={metrics['mask_mAP']:.4f}", flush=True)
        if metrics["mask_mAP"] > best:
            best = metrics["mask_mAP"]
            torch.save({"model": model.state_dict(), "epoch": epoch + 1, "val": metrics}, out / "best.pt")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": epoch + 1, "best": best}, out / "last.pt")
    if run: run.finish()
    best_path = out / "best.pt"
    assert best_path.exists(), f"best.pt tidak ditemukan di {out}"
    (out / "done.json").write_text(json.dumps({
        "model": c["model"], "size": c["run_size"], "weights": str(best_path),
        "sha256": file_sha256(best_path), "best_val_mask_mAP": best,
    }))


def train_rfdetr(c):
    from rfdetr import RFDETRSegSmall
    out = Path(c["out"]); out.mkdir(parents=True, exist_ok=True)
    ckpts = [p for p in out.rglob("*") if p.suffix in (".ckpt", ".pth") and "best" not in p.name]
    resume = str(max(ckpts, key=lambda p: p.stat().st_mtime)) if ckpts else None
    model = RFDETRSegSmall(resolution=c["size"])
    model.train(
        dataset_dir=str(RFDETR_ROOT), output_dir=str(out), epochs=c["epochs"], batch_size=c["batch"],
        grad_accum_steps=max(1, EFFECTIVE_BATCH // c["batch"]), lr=1e-4, num_workers=WORKERS, seed=SEED,
        multi_scale=False, expanded_scales=False,
        early_stopping=True, early_stopping_patience=10, checkpoint_interval=5, resume=resume,
        tensorboard=True, wandb=bool(os.environ.get("WANDB_API_KEY")), project=WANDB_PROJECT,
        run=f"{out.parent.name}-{out.name}", progress_bar=None,
    )
    best = sorted(
        (p for p in out.rglob("*best*") if p.suffix in (".ckpt", ".pth", ".pt")),
        key=lambda p: ("total" not in p.name, -p.stat().st_mtime),
    )
    assert best, f"Checkpoint best tidak ditemukan di {out}"
    (out / "done.json").write_text(json.dumps({
        "model": c["model"], "size": c["run_size"], "weights": str(best[0]),
        "sha256": file_sha256(best[0]), "candidates": [str(p) for p in best],
    }))


TRAINERS = {"yolo26s-seg": train_yolo, "maskrcnn-v2": train_maskrcnn, "rfdetr-seg-small": train_rfdetr}

## 8. Eksekusi Pelatihan (9 Konfigurasi)

Kesembilan run (`3 model x 3 ukuran input`) dijalankan berurutan secara langsung agar log dan progres tampil *real-time* di Colab tanpa risiko *deadlock* akibat `fork()` dengan *thread* W&B aktif. Memori GPU dibersihkan setiap selesai satu konfigurasi, dan run yang sudah memiliki `done.json` valid otomatis dilewati.

In [ ]:
import gc


def run_dir(model, size):
    return OUT / f"size_{size}" / model


CHECKPOINT_CACHE = {}


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def completed_checkpoint(model, size):
    done = run_dir(model, size) / "done.json"
    try:
        record = json.loads(done.read_text())
        if record.get("model") != model or record.get("size") != size:
            return None
        expected = record.get("sha256")
        if not isinstance(expected, str) or len(expected) != 64:
            return None
        weights = Path(record["weights"]).resolve()
        weights.relative_to(run_dir(model, size).resolve())
        if not weights.is_file() or weights.stat().st_size == 0:
            return None
        stat = weights.stat()
        signature = (stat.st_ino, stat.st_size, stat.st_mtime_ns, stat.st_ctime_ns, expected)
        if CHECKPOINT_CACHE.get(weights) != signature:
            if file_sha256(weights) != expected:
                return None
            CHECKPOINT_CACHE[weights] = signature
        return weights
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return None


def train_size(model, size):
    return RFDETR_RES[size] if model.startswith("rfdetr") else size


def record_run(model, size, t0, rc):
    secs = time.time() - t0
    new = not (OUT / "gpu_time.csv").exists()
    with (OUT / "gpu_time.csv").open("a") as f:
        if new:
            f.write("model,size,gpu,started_unix,seconds,returncode\n")
        f.write(f"{model},{size},0,{t0:.0f},{secs:.0f},{rc}\n")
    with (OUT / "LOGBOOK.md").open("a") as f:
        f.write(f"- {time.strftime('%Y-%m-%d %H:%M')} | {model} size={size} gpu=0 rc={rc} {secs / 3600:.2f} h\n")
    print(f"{'OK  ' if rc == 0 else 'FAIL'} {model} {size}: {secs / 60:.1f} min", flush=True)


PENDING = [(m, s) for m in MODELS for s in sorted(SIZES, reverse=True) if completed_checkpoint(m, s) is None]
FAILURES = []
print("Run tertunda:", PENDING)

for model, size in PENDING:
    if completed_checkpoint(model, size) is not None:
        print("skip (sudah selesai):", model, size)
        continue
    d = run_dir(model, size)
    d.mkdir(parents=True, exist_ok=True)
    print(f"\n=== START {model} size={size} (train_size={train_size(model, size)}) ===", flush=True)
    t0, rc = time.time(), 0
    try:
        TRAINERS[model]({
            "model": model, "size": train_size(model, size), "run_size": size,
            "epochs": EPOCHS[model], "batch": BATCH[model][size], "out": str(d),
        })
        if completed_checkpoint(model, size) is None:
            raise RuntimeError("done.json atau checkpoint valid tidak ditemukan")
    except Exception as exc:
        rc = 1
        traceback.print_exc()
        FAILURES.append(f"{model} size={size}: {exc!r}")
    finally:
        try:
            import wandb
            if wandb.run is not None:
                wandb.finish()
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        record_run(model, size, t0, rc)

if FAILURES:
    raise RuntimeError("Training gagal: " + "; ".join(FAILURES))

## 9. Evaluasi *Split* `valid` dan Manifes *Handoff*

Prediksi seluruh model pada `valid` diubah ke format COCO RLE dan dihitung dengan evaluator `pycocotools` (`segm`) yang seragam:
- **`mask mAP@[.5:.95]`**: rata-rata AP di rentang ambang IoU `0.50-0.95` sebagai metrik utama.
- **`AP50` / `AP75`**: AP pada IoU `0.50` dan `0.75` (`AP75` lebih ketat menilai akurasi tepi *mask*).
- **`AP_small` / `AP_medium` / `AP_large`**: performa berdasarkan luas objek dalam piksel.
- **`Waktu inference`**: rata-rata detik per gambar termasuk praproses dan konversi RLE.

In [ ]:
SETS = {"valid": DATASET_ROOT / "valid"}
GT = {}
for name, folder in SETS.items():
    GT[name] = EVAL_OUT / f"gt_{name}.json"
    print(name, "kelas diabaikan:", remap_gt(folder / "_annotations.coco.json", GT[name], NAME_TO_ID))

rows, manifest = [], []
for model in MODELS:
    for size in SIZES:
        weights = completed_checkpoint(model, size)
        if weights is None:
            raise FileNotFoundError(f"Checkpoint valid tidak ditemukan untuk {model} size={size}")
        run = make_predictor(model, str(weights), train_size(model, size), NAME_TO_ID)
        for name, folder in SETS.items():
            t0 = time.time()
            recs = predict_set(run, GT[name], folder)
            (EVAL_OUT / f"pred_{model}_{size}_{name}.json").write_text(json.dumps(recs))
            n_img = len(json.loads(GT[name].read_text())["images"])
            m = coco_mask_eval(GT[name], recs)
            rows.append({"model": model, "size": size, "set": name, **m, "sec_per_img": (time.time() - t0) / n_img})
            manifest.append({
                "model": model, "size": size, "train_size": train_size(model, size),
                "weights": str(weights), "rel_weights": str(weights.relative_to(OUT)),
                "val_mask_mAP": m["mask_mAP"],
            })
            print(rows[-1], flush=True)
        del run; torch.cuda.empty_cache()

RESULTS = pd.DataFrame(rows)
RESULTS.to_csv(OUT / "final_eval.csv", index=False)
(OUT / "handoff_manifest.json").write_text(json.dumps({
    "classes": CATEGORY_NAMES, "name_to_id": NAME_TO_ID, "checkpoints": manifest,
}, indent=2))
display(RESULTS.pivot_table(index="model", columns="size", values="mask_mAP").round(4))

### 9.1 Grafik *Mask* mAP terhadap Ukuran Input dan Biaya GPU

Grafik di bawah menampilkan tren `mask mAP`, perbandingan AP antar-skala objek, serta total jam GPU (`gpu_hours.csv`) di **NVIDIA A100-SXM4-40GB**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
valid_res = RESULTS[RESULTS.set == "valid"]
for model, g in valid_res.groupby("model"):
    g = g.sort_values("size")
    axes[0].plot(g["size"], g["mask_mAP"], marker="o", label=model)
    axes[1].plot(g["size"], g["AP75"], marker="s", label=model)
axes[0].set_title("valid mask mAP@[.5:.95] vs input size")
axes[1].set_title("valid AP75 (sensitivitas tepi mask) vs input size")
for ax in axes:
    ax.set_xlabel("input size"); ax.set_xticks(SIZES); ax.grid(alpha=.3); ax.legend()
axes[0].set_ylabel("score")
plt.tight_layout(); plt.savefig(OUT / "map_vs_size.png", dpi=150); plt.show()

gpu = pd.read_csv(OUT / "gpu_time.csv").groupby(["model", "size"], as_index=False)["seconds"].sum()
gpu["gpu_hours"] = gpu["seconds"] / 3600
display(gpu.round(3)); gpu.to_csv(OUT / "gpu_hours.csv", index=False)

### 9.2 Ringkasan Otomatis dari `final_eval.csv`

Tabel berikut merangkum hasil akhir seluruh konfigurasi pada *split* `valid` langsung dari file CSV hasil eksekusi.

In [ ]:
from IPython.display import Markdown

lines = []
for model, g in RESULTS.groupby("model"):
    for name, h in g.groupby("set"):
        m = h.set_index("size")["mask_mAP"]
        best = m.idxmax()
        d = m.get(960, np.nan) - m.get(640, np.nan)
        t = h.set_index("size")["sec_per_img"]
        lines.append(
            f"- **{model}** pada *{name}*: mAP tertinggi pada {best} px ({m[best]:.3f}); "
            f"selisih 960 - 640 = {d:+.3f}; waktu *inference* {t.min():.2f}–{t.max():.2f} s/gambar."
        )
display(Markdown("\n".join(lines)))

## 10. Analisis Kegagalan Awal (*Split* `valid`)

Lima gambar `valid` dengan AP terendah per model pada resolusi `640` piksel ditampilkan berdampingan dengan *ground truth* untuk melihat kaitannya dengan karakteristik tiap arsitektur:
- **`YOLO26s-seg`**: *mask* dibentuk dari *prototype* beresolusi seperempat input sehingga tepi benda kecil atau tipis mudah kabur.
- **`Mask R-CNN v2`**: *mask* diprediksi di *grid* `28x28` per RoI lalu di-*upsample*, membatasi ketajaman detail pada objek rumit.
- **`RF-DETR-Seg Small`**: *mask* diturunkan dari *query* dan *patch feature* berukuran `12` piksel sehingga objek yang sangat kecil atau berdempetan rapat sulit dipisahkan.

In [ ]:
FAIL_SIZE = 640
FAIL_SET = "valid"
gt = read_json(GT[FAIL_SET])

for FAIL_MODEL in MODELS:
    pred_path = EVAL_OUT / f"pred_{FAIL_MODEL}_{FAIL_SIZE}_{FAIL_SET}.json"
    if not pred_path.exists():
        continue
    recs = json.loads(pred_path.read_text())
    per_img = [
        (im, coco_mask_eval(GT[FAIL_SET], [r for r in recs if r["image_id"] == im["id"]], [im["id"]])["mask_mAP"])
        for im in gt["images"]
    ]
    worst = sorted(per_img, key=lambda t: t[1])[:5]
    fig, axes = plt.subplots(len(worst), 2, figsize=(14, 4.5 * len(worst)), squeeze=False)
    for (im, ap), (a0, a1) in zip(worst, axes):
        rgb = load_rgb(SETS[FAIL_SET] / im["file_name"])
        panels = (
            (a0, [decode(a["segmentation"], im["height"], im["width"]) for a in gt["annotations"] if a["image_id"] == im["id"]], "GT"),
            (a1, [decode(r["segmentation"], im["height"], im["width"]) for r in recs if r["image_id"] == im["id"] and r["score"] >= 0.25],
             f"{FAIL_MODEL} pred@0.25 (AP={ap:.3f})"),
        )
        for ax, masks, title in panels:
            over = rgb.copy()
            for k, m in enumerate(masks):
                sel = m.astype(bool)
                over[sel] = (0.5 * over[sel] + 0.5 * np.array(plt.cm.tab20(k % 20)[:3]) * 255).astype(np.uint8)
            ax.imshow(over); ax.set_title(f"{im['file_name']} | {title} | {len(masks)} masks", fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT / f"failures_{FAIL_MODEL}_{FAIL_SIZE}.png", dpi=110); plt.show()

### 10.1 Uji Lanjutan: Perbandingan pada Ukuran Input Lain

Jika kegagalan dipicu oleh keterbatasan resolusi fitur, skor AP pada gambar yang sama akan naik ketika input diperbesar ke `960` piksel. Bila skor tetap rendah di ketiga ukuran, penyebab dominannya mengarah ke kelangkaan kelas di data latih atau ambiguitas label.

In [ ]:
for FAIL_MODEL in MODELS:
    pred_path = EVAL_OUT / f"pred_{FAIL_MODEL}_{FAIL_SIZE}_{FAIL_SET}.json"
    if not pred_path.exists():
        continue
    recs = json.loads(pred_path.read_text())
    per_img = [
        (im, coco_mask_eval(GT[FAIL_SET], [r for r in recs if r["image_id"] == im["id"]], [im["id"]])["mask_mAP"])
        for im in gt["images"]
    ]
    worst = sorted(per_img, key=lambda t: t[1])[:5]
    preds_by_size = {
        s: json.loads(p.read_text()) for s in SIZES
        if (p := EVAL_OUT / f"pred_{FAIL_MODEL}_{s}_{FAIL_SET}.json").exists()
    }
    rows = []
    for im, _ in worst:
        row = {"model": FAIL_MODEL, "file": im["file_name"]}
        for s, rs in preds_by_size.items():
            row[f"AP@{s}"] = coco_mask_eval(GT[FAIL_SET], [r for r in rs if r["image_id"] == im["id"]], [im["id"]])["mask_mAP"]
        rows.append(row)
    display(pd.DataFrame(rows).round(3))

## 11. Pengemasan Artefak Output

Seluruh bobot terbaik (`best.pt`, `checkpoint_best_total.pth`), grafik evaluasi, `results.csv`, dan `handoff_manifest.json` dikemas ke `instance_seg_colab_outputs.zip` agar mudah dipindahkan ke Google Drive maupun dipakai ulang di notebook pengujian.

In [ ]:
BUNDLE = LOCAL_WORK / "instance_seg_colab_outputs"
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)

for f in ("final_eval.csv", "gpu_time.csv", "gpu_hours.csv", "LOGBOOK.md", "map_vs_size.png", "handoff_manifest.json"):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, BUNDLE / f)
for png in OUT.glob("failures_*.png"):
    shutil.copy2(png, BUNDLE / png.name)
if EVAL_OUT.exists():
    shutil.copytree(EVAL_OUT, BUNDLE / "eval", dirs_exist_ok=True)
for model in MODELS:
    for size in SIZES:
        w = completed_checkpoint(model, size)
        if w is not None:
            dst = BUNDLE / "weights" / f"size_{size}" / model
            dst.mkdir(parents=True, exist_ok=True)
            shutil.copy2(w, dst / w.name)
            for extra in ("results.csv", "done.json"):
                if (run_dir(model, size) / extra).exists():
                    shutil.copy2(run_dir(model, size) / extra, dst / extra)

zip_path = shutil.make_archive(str(OUT.parent / "instance_seg_colab_outputs"), "zip", root_dir=BUNDLE)
print("Arsip siap:", zip_path, f"({Path(zip_path).stat().st_size / 1e6:.1f} MB)")

## 12. Keterbatasan dan Kesimpulan Training

### Keterbatasan
- `RF-DETR-Seg Small` memakai resolusi `312` dan `648` piksel alih-alih tepat `320` dan `640` karena syarat kelipatan `24` pada arsitektur *backbone*.
- Dua gambar `train` dan dua gambar `valid` dari export Roboflow dibuang saat sanitasi awal karena dimensinya pasca-koreksi EXIF tidak selaras dengan koordinat poligon COCO.
- Seluruh model dilatih menggunakan satu *random seed* (`seed = 42`).

### Kesimpulan Tahap Training (`valid` Set, A100-SXM4-40GB)
1. **Pengaruh ukuran input (H1 & H2):** Pada `valid` (mayoritas gambar sumber `~640 px`), menaikkan ukuran dari `320` ke `640` piksel memberi lonjakan tajam pada `RF-DETR-Seg Small` (`0.3169 -> 0.3930`, termasuk kenaikan `AP_small` dari `0.0799` ke `0.2256`) dan `Mask R-CNN v2` (`0.2026 -> 0.2180`). Sebaliknya, perbesaran lanjut dari `640` ke `960` piksel justru stagnan atau sedikit turun pada keduanya (`RF-DETR` `0.3850`; `Mask R-CNN` `0.2069`) karena gambar sumber tidak menyimpan detail tambahan, mengonfirmasi **H2**. Hanya `YOLO26s-seg` yang terus naik ke `960` piksel (`0.2096 -> 0.2486`) karena *prototype mask* (`240x240` di input `960`) mendapat ruang representasi lebih luas.
2. **Perbandingan arsitektur:** `RF-DETR-Seg Small` unggul jauh di seluruh resolusi dengan skor puncak `0.3930` (`AP50 = 0.4932`, `AP75 = 0.4163`) pada `648 px`, disusul `YOLO26s-seg@960` (`0.2486`) dan `Mask R-CNN v2@640` (`0.2180`).
3. **Biaya komputasi dan kesiapan *handoff*:** Seluruh `9` run tuntas dalam `3.29 GPU-jam` (`YOLO26s-seg` `0.80 jam`, `Mask R-CNN v2` `0.83 jam`, dan `RF-DETR-Seg Small` `1.86 jam`). Sembilan *checkpoint* terbaik beserta `handoff_manifest.json` (`74` kelas aktif) telah tersimpan untuk tahap pengujian pada *test set* mandiri di `02_eval_inference_instance_segmentation_colab.ipynb`.